In [ ]:
import sys
sys.path.append('..')

import torch
from src.sae import SparseAutoencoder
from src.training import train_sae, train_multiple_seeds
from src.generators import generate_world_a, generate_world_b

## Choosing the Sparsity Weight (λ)

Swept sparsity_weight over [0.01, 0.03, 0.05, 0.07, 0.1] on World A to find a reasonable tradeoff between reconstruction quality and sparsity. As expected, higher λ improves sparsity at the cost of reconstruction accuracy. Chose **λ = 0.03** as a middle ground: reconstruction loss stays low (~0.07) while sparsity loss drops meaningfully (~40%) from the λ=0.01 baseline.

In [1]:
from src.sae import SparseAutoencoder
for sw in [0.01, 0.03, 0.05, 0.07, 0.1]:
    m = SparseAutoencoder(input_dim=6, latent_dim=20)
    h = train_sae(m, X_tensor, n_epochs=200, sparsity_weight=sw)
    final_recon = h["recon"][-1]
    final_sparsity = h["sparsity"][-1]
    print(f"sparsity_weight={sw}: final_recon={final_recon:.4f}  final_sparsity={final_sparsity:.4f}")

ModuleNotFoundError: No module named 'src'

## Experiment 3: Training Across Multiple Seeds

Training 5 SAEs per world (same data, different random initializations via `torch.manual_seed(seed)` inside `train_multiple_seeds`) to check whether reconstruction and sparsity outcomes are consistent, or just a product of seed luck.

Note: an earlier ad-hoc comparison (outside this function) suggested World A and World B might need different λ values to match, this turned out to be a seeding artifact, since one world's model had been reseeded after a kernel restart and the other hadn't. `train_multiple_seeds` avoids this by explicitly seeding every model it creates, making all comparisons here reproducible.

In [2]:
from src.training import train_multiple_seeds
from src.generators import generate_world_a

results_a, X_tensor_a = train_multiple_seeds(generate_world_a, {}, n_seeds=5)

ModuleNotFoundError: No module named 'src'

## World B - Same Procedure

Training 5 SAEs on World B using the identical procedure and λ=0.03, for 
direct seed-by-seed comparison against World A.

In [ ]:
from src.training import train_multiple_seeds
from src.generators import generate_world_b
results_b, X_tensor_b = train_multiple_seeds(generate_world_b, {}, n_seeds=5)

## Result: Aggregate Statistics Match Closely

Comparing seed-by-seed, World A and World B show closely matched reconstruction loss and active-latent counts across all 5 seeds - differences between worlds are much smaller than the natural seed-to-seed variance within either world. 
This is consistent with **H0 (statistical hypothesis)**: no evidence yet that causal structure affects these aggregate training outcomes.

This doesn't resolve the core research question, though - these are coarse, aggregate statistics. Experiment 4 will check whether the *specific* latents learned actually correspond to A/B the same way, and whether the internal representation geometry differs between worlds.